<center>

# [Компьютерное зрение](http://rairi.ru/wiki/index.php/%D0%9A%D0%BE%D0%BC%D0%BF%D1%8C%D1%8E%D1%82%D0%B5%D1%80%D0%BD%D0%BE%D0%B5_%D0%B7%D1%80%D0%B5%D0%BD%D0%B8%D0%B5)

## <center> Семинар 12 - MinkowskiEngine

<a target="_blank" href="https://colab.research.google.com/github/alexmelekhin/cv_course_2023/blob/main/seminars/seminar_12/Seminar_12.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

***

# Установка MinkowskiEngine (Colab и локально)

Официальный репозиторий и wiki: [MinkowskiEngine](https://github.com/NVIDIA/MinkowskiEngine), [Installation](https://github.com/NVIDIA/MinkowskiEngine/wiki/Installation).

**Важно**

1. **Сначала PyTorch**, затем сборка ME — `setup.py` импортирует `torch` на этапе *get_requires*. У современного `pip` изолированное окружение сборки **не видит** уже установленный torch → ошибка `Pytorch not found`. Решение: флаг **`--no-build-isolation`** (см. ячейку ниже).
2. **Не фиксируйте** устаревший `torch==1.13.1`: на новых Python колёса с этой версией отсутствуют. Ставьте актуальный torch с [pytorch.org](https://pytorch.org) под вашу CUDA (или используйте torch в Colab).
3. **`numpy.distutils`** в `setup.py` ME — для поиска BLAS нужен **NumPy 1.x** (`numpy<2`) на время сборки. Также нужен **`setuptools` 59.x** (`setuptools>=59.5,<60`): при setuptools ≥ 60 пропадает `distutils.msvccompiler` → ошибка при *Preparing metadata*.
4. **Python 3.12+** (в т.ч. 3.13): сборка часто ломается или не поддерживается; надёжнее отдельное окружение **`python=3.10` или `3.11`** (conda), GCC, `libopenblas-dev` (Linux).
5. **CUDA**: версия toolkit / `CUDA_HOME` должна соответствовать сборке PyTorch; при `nvcc not found` ME соберётся в **CPU-only** (если `torch.cuda.is_available()` ложно).

Если сборка всё равно падает, см. [issue #621 — troubleshooting (CUDA 11/12)](https://github.com/NVIDIA/MinkowskiEngine/issues/621).

Дальше выполните ячейки по порядку: PyTorch → зависимости → сборка ME → проверка импорта.

In [1]:
# PyTorch: установите под вашу систему (https://pytorch.org). Не используйте устаревший pin torch==1.13.1.
# В Google Colab с GPU torch обычно уже есть — ячейку можно пропустить.
# Примеры (раскомментируйте одну строку при необходимости):
# !pip install -q torch torchvision
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124

import sys

try:
    import torch
except ImportError as e:
    err = str(e)
    if "iJIT_NotifyEvent" in err or "undefined symbol" in err:
        raise SystemExit(
            "Ошибка iJIT_NotifyEvent: несовместимость conda PyTorch с MKL ≥ 2024.1. "
            "В терминале: conda install -y 'mkl<2024.1' 'intel-openmp<2024.1' "
            "или torch через pip с pytorch.org. См. https://github.com/pytorch/pytorch/issues/123097"
        ) from e
    raise SystemExit(
        "Сначала установите torch (см. комментарии выше или pip install torch torchvision)."
    ) from e

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if sys.version_info >= (3, 12):
    print(
        "Предупреждение: для MinkowskiEngine рекомендуется Python 3.10–3.11 (conda). "
        "На 3.12+ сборка часто не проходит."
    )

torch: 2.5.1 | CUDA available: True


In [2]:
# Инструменты сборки + NumPy 1.x (setup.py ME использует numpy.distutils, удалённый в NumPy 2)
# setuptools >= 60: нет distutils.msvccompiler → ошибка Preparing metadata для ME
import sys
import subprocess

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "wheel", "ninja", "packaging"]
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "setuptools>=59.5,<60"]
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "numpy>=1.21,<2"]
)


0

In [ ]:
import os
import sys
import shutil
import subprocess

import torch

print(f"torch {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Подсказка по CUDA_HOME (сборка CUDA-расширений ME)
if torch.cuda.is_available() and "CUDA_HOME" not in os.environ:
    try:
        from torch.utils.cpp_extension import _find_cuda_home

        cuda_home = _find_cuda_home()
        if cuda_home:
            os.environ["CUDA_HOME"] = cuda_home
            print("CUDA_HOME (auto):", cuda_home)
    except Exception as exc:
        print("Не удалось выставить CUDA_HOME автоматически:", exc)

# Системные пакеты (OpenBLAS): Colab — без sudo; локально при отсутствии — сообщение
if sys.platform.startswith("linux") and shutil.which("apt-get"):
    if "google.colab" in sys.modules:
        subprocess.run(
            "apt-get update -qq && apt-get install -y -qq libopenblas-dev build-essential",
            shell=True,
            check=False,
        )
    else:
        print("При ошибках BLAS при сборке: sudo apt install libopenblas-dev build-essential")

os.environ.setdefault("MAX_JOBS", "4")

try:
    import MinkowskiEngine  # noqa: F401

    print("MinkowskiEngine уже установлен — пропускаем повторную сборку pip.")
except ImportError:
    # --no-build-isolation: pip должен видеть установленный torch при чтении setup.py
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "git+https://github.com/NVIDIA/MinkowskiEngine",
        "-v",
        "--no-deps",
        "--no-build-isolation",
    ]
    print("Запуск:", " ".join(install_cmd))
    subprocess.run(install_cmd, check=False)

In [ ]:
import torch

print(f"Is CUDA available in torch?: {torch.cuda.is_available()}")
try:
    import MinkowskiEngine as ME
except ImportError as e:
    raise ImportError(
        "Не удалось импортировать MinkowskiEngine. Проверьте: "
        "(1) ячейка сборки завершилась без ошибок; "
        "(2) используется Python 3.10–3.11; "
        "(3) torch установлен до ME; "
        "(4) флаг --no-build-isolation при установке. "
        "См. https://github.com/NVIDIA/MinkowskiEngine/issues/628"
    ) from e

print(f"MinkowskiEngine CUDA: {ME.is_cuda_available()}")
ME.print_diagnostics()

# ModelNet40-классификатор на MinkowskiEngine

Предлагается ознакомиться с оффициальными туториалами:

https://nvidia.github.io/MinkowskiEngine/demo/training.html

https://github.com/NVIDIA/MinkowskiEngine/blob/master/examples/training.py

https://nvidia.github.io/MinkowskiEngine/demo/modelnet40_classification.html


## Домашнее задание

Обучить простейшую модель классификации 3D объектов на датасете ModelNet40, воспользовавшись туториалом выше и кодом отсюда: https://github.com/NVIDIA/MinkowskiEngine/blob/master/examples/classification_modelnet40.py 

Вам предлагается обучить простейшую модель `minkfcnn` (аргумент `--network minkfcnn`).

В качестве отчета по заданию вам предлагается приложить результаты обучения (логи).

In [ ]:
!pip install -q h5py scikit-learn matplotlib

In [ ]:
import glob
import json
import os
import random
import subprocess
from types import SimpleNamespace

import h5py
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics as metrics
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import MinkowskiEngine as ME


def seed_all(random_seed: int) -> None:
    torch.manual_seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(random_seed)
        torch.cuda.manual_seed_all(random_seed)
    np.random.seed(random_seed)
    random.seed(random_seed)


def minkowski_collate_fn(list_data):
    coordinates_batch, features_batch, labels_batch = ME.utils.sparse_collate(
        [d["coordinates"] for d in list_data],
        [d["features"] for d in list_data],
        [d["label"] for d in list_data],
        dtype=torch.float32,
    )
    return {
        "coordinates": coordinates_batch,
        "features": features_batch,
        "labels": labels_batch,
    }


MODELNET40_ZIP_MIRRORS = [
    "https://huggingface.co/datasets/Msun/modelnet40/resolve/main/modelnet40_ply_hdf5_2048.zip",
    "https://shapenet.cs.stanford.edu/media/modelnet40_ply_hdf5_2048.zip",
]


def download_modelnet40_dataset():
    import shutil as _shutil

    zip_name = "modelnet40_ply_hdf5_2048.zip"
    data_dir = "modelnet40_ply_hdf5_2048"
    env_url = (os.environ.get("MODELNET40_URL") or "").strip()
    if env_url:
        urls = [env_url] + [u for u in MODELNET40_ZIP_MIRRORS if u != env_url]
    else:
        urls = list(MODELNET40_ZIP_MIRRORS)
    h5_ok = os.path.isdir(data_dir) and len(glob.glob(os.path.join(data_dir, "ply_data_*.h5"))) > 0
    if h5_ok:
        return
    if not os.path.exists(zip_name) or os.path.getsize(zip_name) < 1_000_000:
        if os.path.exists(zip_name):
            os.remove(zip_name)
        print("Downloading ModelNet40 (2k points)...")
        last = None
        for url in urls:
            try:
                print("Trying:", url[:90] + ("..." if len(url) > 90 else ""))
                if _shutil.which("wget"):
                    subprocess.run(
                        ["wget", "-O", zip_name, "--tries=3", "--timeout=120", "--no-check-certificate", url],
                        check=True,
                    )
                elif _shutil.which("curl"):
                    subprocess.run(
                        ["curl", "-fL", "--retry", "3", "--connect-timeout", "30", "-o", zip_name, url],
                        check=True,
                    )
                else:
                    raise RuntimeError("Нужен wget или curl для загрузки")
                if os.path.getsize(zip_name) > 1_000_000:
                    break
                os.remove(zip_name)
            except (OSError, subprocess.CalledProcessError) as e:
                last = e
                print("Не удалось:", e)
                if os.path.exists(zip_name):
                    try:
                        os.remove(zip_name)
                    except OSError:
                        pass
        else:
            raise RuntimeError(
                "Не удалось скачать архив ни с одного зеркала. "
                "Укажите MODELNET40_URL или положите modelnet40_ply_hdf5_2048.zip вручную."
            ) from last
    if not os.path.isdir(data_dir):
        subprocess.run(["unzip", "-o", zip_name], check=True)


class ModelNet40H5(Dataset):
    def __init__(
        self,
        phase: str,
        data_root: str = "modelnet40_ply_hdf5_2048",
        transform=None,
        num_points: int = 2048,
    ):
        super().__init__()
        download_modelnet40_dataset()
        phase = "test" if phase in ("val", "test") else "train"
        self.data, self.label = self._load_data(data_root, phase)
        self.transform = transform
        self.phase = phase
        self.num_points = num_points

    def _load_data(self, data_root, phase):
        data, labels = [], []
        assert os.path.exists(data_root), f"{data_root} does not exist"
        files = glob.glob(os.path.join(data_root, f"ply_data_{phase}*.h5"))
        assert len(files) > 0, "No h5 files found"
        for h5_name in files:
            with h5py.File(h5_name) as f:
                data.extend(f["data"][:].astype("float32"))
                labels.extend(f["label"][:].astype("int64"))
        return np.stack(data, axis=0), np.stack(labels, axis=0)

    def __getitem__(self, i: int) -> dict:
        xyz = self.data[i]
        if self.phase == "train":
            np.random.shuffle(xyz)
        if len(xyz) > self.num_points:
            xyz = xyz[: self.num_points]
        if self.transform is not None:
            xyz = self.transform(xyz)
        label = self.label[i]
        xyz = torch.from_numpy(xyz).to(torch.float32)
        label = torch.from_numpy(label)
        return {"coordinates": xyz, "features": xyz, "label": label}

    def __len__(self):
        return self.data.shape[0]


class CoordinateTransformation:
    def __init__(self, scale_range=(0.9, 1.1), trans=0.25, jitter=0.025, clip=0.05):
        self.scale_range = scale_range
        self.trans = trans
        self.jitter = jitter
        self.clip = clip

    def __call__(self, coords):
        if random.random() < 0.9:
            coords = coords * np.random.uniform(
                low=self.scale_range[0], high=self.scale_range[1], size=[1, 3]
            )
        if random.random() < 0.9:
            coords = coords + np.random.uniform(low=-self.trans, high=self.trans, size=[1, 3])
        if random.random() < 0.7:
            coords = coords + np.clip(
                self.jitter * (np.random.rand(len(coords), 3) - 0.5),
                -self.clip,
                self.clip,
            )
        return coords


class CoordinateTranslation:
    def __init__(self, translation: float):
        self.trans = translation

    def __call__(self, coords):
        if self.trans > 0:
            coords = coords + np.random.uniform(low=-self.trans, high=self.trans, size=[1, 3])
        return coords


def make_data_loader(phase, config):
    assert phase in ("train", "val", "test")
    is_train = phase == "train"
    dataset = ModelNet40H5(
        phase=phase,
        transform=CoordinateTransformation(trans=config.translation)
        if is_train
        else CoordinateTranslation(config.test_translation),
    )
    return DataLoader(
        dataset,
        num_workers=config.num_workers,
        shuffle=is_train,
        collate_fn=minkowski_collate_fn,
        batch_size=config.batch_size,
    )


def create_input_batch(batch, device, quantization_size):
    batch["coordinates"][:, 1:] = batch["coordinates"][:, 1:] / quantization_size
    return ME.TensorField(
        coordinates=batch["coordinates"],
        features=batch["features"],
        device=device,
    )


def criterion(pred, labels, smoothing=True):
    labels = labels.contiguous().view(-1)
    if smoothing:
        eps = 0.2
        n_class = pred.size(1)
        one_hot = torch.zeros_like(pred).scatter(1, labels.view(-1, 1), 1)
        one_hot = one_hot * (1 - eps) + (1 - one_hot) * eps / (n_class - 1)
        log_prb = F.log_softmax(pred, dim=1)
        return -(one_hot * log_prb).sum(dim=1).mean()
    return F.cross_entropy(pred, labels, reduction="mean")

In [ ]:
class MinkowskiFCNN(ME.MinkowskiNetwork):
    def __init__(
        self,
        in_channel,
        out_channel,
        embedding_channel=1024,
        channels=(32, 48, 64, 96, 128),
        D=3,
    ):
        ME.MinkowskiNetwork.__init__(self, D)
        self.network_initialization(
            in_channel,
            out_channel,
            channels=channels,
            embedding_channel=embedding_channel,
            kernel_size=3,
            D=D,
        )
        self.weight_initialization()

    def get_mlp_block(self, in_channel, out_channel):
        return nn.Sequential(
            ME.MinkowskiLinear(in_channel, out_channel, bias=False),
            ME.MinkowskiBatchNorm(out_channel),
            ME.MinkowskiLeakyReLU(),
        )

    def get_conv_block(self, in_channel, out_channel, kernel_size, stride):
        return nn.Sequential(
            ME.MinkowskiConvolution(
                in_channel,
                out_channel,
                kernel_size=kernel_size,
                stride=stride,
                dimension=self.D,
            ),
            ME.MinkowskiBatchNorm(out_channel),
            ME.MinkowskiLeakyReLU(),
        )

    def network_initialization(
        self,
        in_channel,
        out_channel,
        channels,
        embedding_channel,
        kernel_size,
        D=3,
    ):
        self.mlp1 = self.get_mlp_block(in_channel, channels[0])
        self.conv1 = self.get_conv_block(channels[0], channels[1], kernel_size=kernel_size, stride=1)
        self.conv2 = self.get_conv_block(channels[1], channels[2], kernel_size=kernel_size, stride=2)
        self.conv3 = self.get_conv_block(channels[2], channels[3], kernel_size=kernel_size, stride=2)
        self.conv4 = self.get_conv_block(channels[3], channels[4], kernel_size=kernel_size, stride=2)
        self.conv5 = nn.Sequential(
            self.get_conv_block(
                channels[1] + channels[2] + channels[3] + channels[4],
                embedding_channel // 4,
                kernel_size=3,
                stride=2,
            ),
            self.get_conv_block(
                embedding_channel // 4,
                embedding_channel // 2,
                kernel_size=3,
                stride=2,
            ),
            self.get_conv_block(
                embedding_channel // 2,
                embedding_channel,
                kernel_size=3,
                stride=2,
            ),
        )
        self.pool = ME.MinkowskiMaxPooling(kernel_size=3, stride=2, dimension=D)
        self.global_max_pool = ME.MinkowskiGlobalMaxPooling()
        self.global_avg_pool = ME.MinkowskiGlobalAvgPooling()
        self.final = nn.Sequential(
            self.get_mlp_block(embedding_channel * 2, 512),
            ME.MinkowskiDropout(),
            self.get_mlp_block(512, 512),
            ME.MinkowskiLinear(512, out_channel, bias=True),
        )

    def weight_initialization(self):
        for m in self.modules():
            if isinstance(m, ME.MinkowskiConvolution):
                ME.utils.kaiming_normal_(m.kernel, mode="fan_out", nonlinearity="relu")
            if isinstance(m, ME.MinkowskiBatchNorm):
                nn.init.constant_(m.bn.weight, 1)
                nn.init.constant_(m.bn.bias, 0)

    def forward(self, x: ME.TensorField):
        x = self.mlp1(x)
        y = x.sparse()
        y = self.conv1(y)
        y1 = self.pool(y)
        y = self.conv2(y1)
        y2 = self.pool(y)
        y = self.conv3(y2)
        y3 = self.pool(y)
        y = self.conv4(y3)
        y4 = self.pool(y)
        x1 = y1.slice(x)
        x2 = y2.slice(x)
        x3 = y3.slice(x)
        x4 = y4.slice(x)
        x = ME.cat(x1, x2, x3, x4)
        y = self.conv5(x.sparse())
        x1 = self.global_max_pool(y)
        x2 = self.global_avg_pool(y)
        return self.final(ME.cat(x1, x2)).F

In [ ]:
def evaluate(net, device, config):
    """Точность на тестовом сплите ModelNet40 (как в classification_modelnet40.py)."""
    data_loader = make_data_loader("test", config)
    net.eval()
    labels, preds = [], []
    with torch.no_grad():
        for batch in data_loader:
            inp = create_input_batch(
                batch,
                device=device,
                quantization_size=config.voxel_size,
            )
            logit = net(inp)
            pred = torch.argmax(logit, 1)
            labels.append(batch["labels"].cpu().numpy())
            preds.append(pred.cpu().numpy())
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return float(metrics.accuracy_score(np.concatenate(labels), np.concatenate(preds)))


def plot_and_save_metrics(history: dict, out_dir: str) -> str:
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    if history["train_step"]:
        ax_loss.plot(history["train_step"], history["train_loss"], label="train loss")
    ax_loss.set_xlabel("iteration")
    ax_loss.set_ylabel("loss")
    ax_loss.set_title("Training loss")
    ax_loss.grid(True, alpha=0.3)
    ax_loss.legend()
    if history["val_step"]:
        ax_acc.plot(history["val_step"], history["val_acc"], "o-", label="val acc")
    ax_acc.set_xlabel("iteration")
    ax_acc.set_ylabel("accuracy")
    ax_acc.set_title("Validation accuracy (test split)")
    ax_acc.set_ylim(0.0, 1.0)
    ax_acc.grid(True, alpha=0.3)
    ax_acc.legend()
    fig.tight_layout()
    path = os.path.join(out_dir, "metrics.png")
    fig.savefig(path, dpi=150)
    plt.show()
    plt.close(fig)
    return path


def train_with_metrics(net, device, config, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    optimizer = optim.SGD(
        net.parameters(),
        lr=config.lr,
        momentum=0.9,
        weight_decay=config.weight_decay,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.max_steps)
    history = {
        "train_step": [],
        "train_loss": [],
        "val_step": [],
        "val_acc": [],
    }
    train_iter = iter(make_data_loader("train", config))
    best_metric = 0.0
    best_step = -1
    net.train()
    for i in range(config.max_steps):
        optimizer.zero_grad()
        try:
            data_dict = next(train_iter)
        except StopIteration:
            train_iter = iter(make_data_loader("train", config))
            data_dict = next(train_iter)
        inp = create_input_batch(
            data_dict,
            device=device,
            quantization_size=config.voxel_size,
        )
        logit = net(inp)
        loss = criterion(logit, data_dict["labels"].to(device))
        loss.backward()
        optimizer.step()
        scheduler.step()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if i % config.stat_freq == 0:
            print(f"Iter {i}: loss={loss.item():.4e}")
            history["train_step"].append(i)
            history["train_loss"].append(float(loss.item()))

        if i % config.val_freq == 0 and i > 0:
            ckpt_path = os.path.join(out_dir, "checkpoint_last.pth")
            torch.save(
                {
                    "state_dict": net.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "curr_iter": i,
                    "history": dict(history),
                },
                ckpt_path,
            )
            acc = evaluate(net, device, config)
            history["val_step"].append(i)
            history["val_acc"].append(acc)
            print(f"Iter {i}: val_acc={acc:.4f} (best was {best_metric:.4f} @ {best_step})")
            if acc > best_metric:
                best_metric = acc
                best_step = i
                torch.save(
                    {
                        "state_dict": net.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(),
                        "curr_iter": i,
                        "best_val_acc": best_metric,
                    },
                    os.path.join(out_dir, "best_model.pth"),
                )
            net.train()

    with open(os.path.join(out_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump({**history, "config": dict(vars(config))}, f, indent=2)
    plot_path = plot_and_save_metrics(history, out_dir)
    print(f"Saved history.json, plot: {plot_path}, best_model.pth in {out_dir}")
    return history, best_metric, best_step


# Параметры: для полного прогона увеличьте max_steps (например 100000) как в официальном примере
OUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "seminar_12_runs"))
config = SimpleNamespace(
    voxel_size=0.05,
    max_steps=5000,
    val_freq=500,
    stat_freq=50,
    batch_size=32,
    lr=0.1,
    weight_decay=1e-4,
    num_workers=2,
    translation=0.2,
    test_translation=0.0,
    seed=777,
)

seed_all(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, OUT_DIR={OUT_DIR}")

net = MinkowskiFCNN(in_channel=3, out_channel=40, embedding_channel=1024).to(device)
print(net)

history, best_acc, best_step = train_with_metrics(net, device, config, OUT_DIR)
test_acc = evaluate(net, device, config)
print(f"Final test accuracy: {test_acc:.4f} | best val acc: {best_acc:.4f} @ step {best_step}")

В рамках результатов обучения прикрепляю логи из файла `seminars/seminar_12/minkowski_training_docker/runs/logs/training.log`

2026-05-13 09:51:26 [INFO] Logging to /logs/training.log
2026-05-13 09:51:26 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-13 09:51:26 [INFO] torch 2.6.0+cu124 cuda=True ME.cuda=False
2026-05-13 09:51:26 [INFO] Start training: max_steps=10000
2026-05-13 09:51:27 [INFO] Downloading ModelNet40 zip...
2026-05-13 10:12:29 [INFO] Logging to /logs/training.log
2026-05-13 10:12:29 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-13 10:12:29 [INFO] torch 2.6.0+cu124 cuda=True ME.cuda=False
2026-05-13 10:12:29 [INFO] Start training: max_steps=10000
2026-05-13 10:12:30 [INFO] Downloading ModelNet40 zip (пробуем зеркала)...
2026-05-13 10:12:30 [INFO] URL: https://huggingface.co/datasets/Msun/modelnet40/resolve/main/modelnet40_ply_hdf5_2048.zip
2026-05-13 10:12:40 [INFO] Extracting ModelNet40 into /workspace
2026-05-13 10:17:26 [INFO] Logging to /logs/training.log
2026-05-13 10:17:26 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-13 10:17:26 [INFO] torch 2.6.0+cu124 cuda=True ME.cuda=False
2026-05-13 10:17:26 [INFO] Start training: max_steps=10000
2026-05-13 10:19:19 [INFO] Logging to /logs/training.log
2026-05-13 10:19:19 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-13 10:19:19 [INFO] torch 2.6.0+cu124 cuda=True ME.cuda=False
2026-05-13 10:19:19 [INFO] Start training: max_steps=10000
2026-05-13 10:29:29 [INFO] Logging to /logs/training.log
2026-05-13 10:29:29 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-13 10:29:29 [INFO] torch 2.12.0+cpu cuda=False ME.cuda=False
2026-05-13 10:29:29 [INFO] Start training: max_steps=5
2026-05-13 10:29:34 [INFO] iter=0 loss=3.9566e+00 elapsed_s=2
2026-05-13 10:29:36 [INFO] iter=1 loss=4.0667e+00 elapsed_s=3
2026-05-13 10:29:37 [INFO] iter=2 loss=4.5003e+00 elapsed_s=5
2026-05-13 10:29:39 [INFO] iter=3 loss=4.6218e+00 elapsed_s=7
2026-05-13 10:35:03 [INFO] iter=3 val_acc=0.0077 best=0.0000 @ -1
2026-05-13 10:35:04 [INFO] iter=4 loss=4.9770e+00 elapsed_s=332
2026-05-13 10:35:05 [INFO] Saved history.json, /workspace/seminar_12_runs/metrics.png, best_model.pth in /workspace/seminar_12_runs
2026-05-13 11:21:13 [INFO] Logging to /logs/training.log
2026-05-13 11:21:13 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-13 11:21:13 [INFO] torch 2.12.0+cpu cuda=False ME.cuda=False
2026-05-13 11:21:13 [INFO] Start training: max_steps=500
2026-05-13 11:21:21 [INFO] iter=0 loss=3.8745e+00 elapsed_s=5
2026-05-13 11:23:42 [INFO] iter=25 loss=9.4743e+00 elapsed_s=145
2026-05-13 11:26:30 [INFO] iter=50 loss=4.4522e+00 elapsed_s=314
2026-05-13 11:27:53 [INFO] iter=75 loss=3.6687e+00 elapsed_s=397
2026-05-13 11:29:11 [INFO] iter=100 loss=3.7626e+00 elapsed_s=475
2026-05-13 11:35:29 [INFO] iter=100 val_acc=0.0393 best=0.0000 @ -1
2026-05-13 11:37:09 [INFO] iter=125 loss=3.4180e+00 elapsed_s=953
2026-05-13 11:38:41 [INFO] iter=150 loss=3.5715e+00 elapsed_s=1045
2026-05-13 11:40:16 [INFO] iter=175 loss=3.6922e+00 elapsed_s=1139
2026-05-13 11:42:05 [INFO] iter=200 loss=3.4180e+00 elapsed_s=1249
2026-05-13 11:49:10 [INFO] iter=200 val_acc=0.0401 best=0.0393 @ 100
2026-05-13 11:50:48 [INFO] iter=225 loss=3.5356e+00 elapsed_s=1772
2026-05-13 11:52:40 [INFO] iter=250 loss=3.7158e+00 elapsed_s=1884
2026-05-13 11:54:14 [INFO] iter=275 loss=3.4383e+00 elapsed_s=1977
2026-05-13 11:56:14 [INFO] iter=300 loss=3.4655e+00 elapsed_s=2098
2026-05-13 12:03:26 [INFO] iter=300 val_acc=0.0405 best=0.0401 @ 200
2026-05-13 12:05:04 [INFO] iter=325 loss=3.5890e+00 elapsed_s=2628
2026-05-13 12:07:00 [INFO] iter=350 loss=3.5743e+00 elapsed_s=2744
2026-05-13 12:08:31 [INFO] iter=375 loss=3.3966e+00 elapsed_s=2835
2026-05-13 12:10:29 [INFO] iter=400 loss=3.6366e+00 elapsed_s=2952
2026-05-13 12:15:02 [INFO] iter=400 val_acc=0.0405 best=0.0405 @ 300
2026-05-13 12:16:03 [INFO] iter=425 loss=3.5213e+00 elapsed_s=3287
2026-05-13 12:17:06 [INFO] iter=450 loss=3.4938e+00 elapsed_s=3349
2026-05-13 12:17:45 [INFO] iter=475 loss=3.5256e+00 elapsed_s=3389
2026-05-13 12:18:24 [INFO] Saved history.json, /workspace/seminar_12_runs/metrics.png, best_model.pth in /workspace/seminar_12_runs
2026-05-13 12:21:19 [INFO] Final test_acc=0.0409 best_val=0.0405 @ step 300
2026-05-13 14:36:48 [INFO] Logging to /logs/training.log
2026-05-13 14:36:48 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-13 14:36:48 [INFO] torch 2.12.0+cpu cuda=False ME.cuda=False
2026-05-13 14:36:49 [INFO] Start training: max_steps=5000
2026-05-13 14:36:54 [INFO] iter=0 loss=3.7572e+00 elapsed_s=2
2026-05-13 14:37:33 [INFO] iter=25 loss=7.1297e+00 elapsed_s=41
2026-05-13 14:38:12 [INFO] iter=50 loss=4.4015e+00 elapsed_s=80
2026-05-13 14:38:51 [INFO] iter=75 loss=3.5021e+00 elapsed_s=119
2026-05-13 14:39:30 [INFO] iter=100 loss=3.4892e+00 elapsed_s=158
2026-05-13 14:42:23 [INFO] iter=100 val_acc=0.0385 best=0.0000 @ -1
2026-05-13 14:43:03 [INFO] iter=125 loss=3.1980e+00 elapsed_s=371
2026-05-13 14:43:43 [INFO] iter=150 loss=3.6156e+00 elapsed_s=411
2026-05-13 14:44:22 [INFO] iter=175 loss=3.6867e+00 elapsed_s=450
2026-05-13 14:45:00 [INFO] iter=200 loss=3.4534e+00 elapsed_s=488
2026-05-13 14:47:54 [INFO] iter=200 val_acc=0.0429 best=0.0385 @ 100
2026-05-13 14:48:35 [INFO] iter=225 loss=3.4201e+00 elapsed_s=703
2026-05-13 14:49:15 [INFO] iter=250 loss=3.7196e+00 elapsed_s=743
2026-05-13 14:49:53 [INFO] iter=275 loss=3.3590e+00 elapsed_s=781
2026-05-13 14:50:33 [INFO] iter=300 loss=3.4421e+00 elapsed_s=821
2026-05-13 14:53:27 [INFO] iter=300 val_acc=0.0413 best=0.0429 @ 200
2026-05-13 14:54:06 [INFO] iter=325 loss=3.5945e+00 elapsed_s=1034
2026-05-13 14:54:45 [INFO] iter=350 loss=3.6786e+00 elapsed_s=1073
2026-05-13 14:55:23 [INFO] iter=375 loss=3.4204e+00 elapsed_s=1111
2026-05-13 14:56:02 [INFO] iter=400 loss=3.4880e+00 elapsed_s=1150
2026-05-13 14:58:53 [INFO] iter=400 val_acc=0.0409 best=0.0429 @ 200
2026-05-13 14:59:33 [INFO] iter=425 loss=3.6333e+00 elapsed_s=1361
2026-05-13 15:00:12 [INFO] iter=450 loss=3.4802e+00 elapsed_s=1400
2026-05-13 15:00:50 [INFO] iter=475 loss=3.6380e+00 elapsed_s=1438
2026-05-13 15:01:29 [INFO] iter=500 loss=3.6316e+00 elapsed_s=1477
2026-05-13 15:04:21 [INFO] iter=500 val_acc=0.0409 best=0.0429 @ 200
2026-05-13 15:05:01 [INFO] iter=525 loss=3.6653e+00 elapsed_s=1689
2026-05-13 15:05:41 [INFO] iter=550 loss=3.7238e+00 elapsed_s=1729
2026-05-13 15:06:21 [INFO] iter=575 loss=3.5286e+00 elapsed_s=1769
2026-05-13 15:07:00 [INFO] iter=600 loss=3.5090e+00 elapsed_s=1808
2026-05-13 15:09:55 [INFO] iter=600 val_acc=0.0405 best=0.0429 @ 200
2026-05-13 15:10:36 [INFO] iter=625 loss=3.6232e+00 elapsed_s=2024
2026-05-13 15:11:14 [INFO] iter=650 loss=3.7974e+00 elapsed_s=2062
2026-05-13 15:11:53 [INFO] iter=675 loss=3.6993e+00 elapsed_s=2101
2026-05-13 15:12:32 [INFO] iter=700 loss=3.5196e+00 elapsed_s=2139
2026-05-13 15:15:31 [INFO] iter=700 val_acc=0.0409 best=0.0429 @ 200
2026-05-13 15:16:11 [INFO] iter=725 loss=3.7636e+00 elapsed_s=2359
2026-05-13 15:16:51 [INFO] iter=750 loss=3.6504e+00 elapsed_s=2399
2026-05-13 15:17:32 [INFO] iter=775 loss=3.5569e+00 elapsed_s=2440
2026-05-13 15:18:13 [INFO] iter=800 loss=3.6423e+00 elapsed_s=2481
2026-05-13 15:21:12 [INFO] iter=800 val_acc=0.0413 best=0.0429 @ 200
2026-05-13 15:21:52 [INFO] iter=825 loss=3.4549e+00 elapsed_s=2700
2026-05-13 15:22:31 [INFO] iter=850 loss=3.8336e+00 elapsed_s=2738
2026-05-13 15:23:10 [INFO] iter=875 loss=3.6149e+00 elapsed_s=2778
2026-05-13 15:23:50 [INFO] iter=900 loss=3.7931e+00 elapsed_s=2818
2026-05-13 15:26:46 [INFO] iter=900 val_acc=0.0377 best=0.0429 @ 200
2026-05-13 15:27:26 [INFO] iter=925 loss=3.3620e+00 elapsed_s=3034
2026-05-13 15:28:05 [INFO] iter=950 loss=3.5248e+00 elapsed_s=3073
2026-05-13 15:28:44 [INFO] iter=975 loss=3.5200e+00 elapsed_s=3112
2026-05-13 15:29:25 [INFO] iter=1000 loss=3.3547e+00 elapsed_s=3153
2026-05-13 15:32:23 [INFO] iter=1000 val_acc=0.0421 best=0.0429 @ 200
2026-05-13 15:33:03 [INFO] iter=1025 loss=3.8475e+00 elapsed_s=3371
2026-05-13 15:33:41 [INFO] iter=1050 loss=3.4490e+00 elapsed_s=3409
2026-05-13 15:34:20 [INFO] iter=1075 loss=3.4477e+00 elapsed_s=3448
2026-05-13 15:34:59 [INFO] iter=1100 loss=3.5099e+00 elapsed_s=3487
2026-05-13 15:37:58 [INFO] iter=1100 val_acc=0.0413 best=0.0429 @ 200
2026-05-13 15:38:37 [INFO] iter=1125 loss=3.7300e+00 elapsed_s=3704
2026-05-13 15:39:15 [INFO] iter=1150 loss=4.0220e+00 elapsed_s=3742
2026-05-13 15:39:55 [INFO] iter=1175 loss=3.6466e+00 elapsed_s=3783
2026-05-13 15:40:35 [INFO] iter=1200 loss=3.4319e+00 elapsed_s=3823
2026-05-13 15:43:41 [INFO] iter=1200 val_acc=0.0405 best=0.0429 @ 200
2026-05-13 15:44:21 [INFO] iter=1225 loss=3.4435e+00 elapsed_s=4049
2026-05-13 15:45:11 [INFO] iter=1250 loss=3.3356e+00 elapsed_s=4099
2026-05-13 15:45:52 [INFO] iter=1275 loss=3.7082e+00 elapsed_s=4140
2026-05-13 15:46:32 [INFO] iter=1300 loss=3.5987e+00 elapsed_s=4180
2026-05-13 15:49:36 [INFO] iter=1300 val_acc=0.0405 best=0.0429 @ 200
2026-05-13 15:50:17 [INFO] iter=1325 loss=3.7278e+00 elapsed_s=4405
2026-05-13 15:50:58 [INFO] iter=1350 loss=3.6426e+00 elapsed_s=4446
2026-05-13 15:51:38 [INFO] iter=1375 loss=3.1326e+00 elapsed_s=4485
2026-05-13 15:52:18 [INFO] iter=1400 loss=3.6748e+00 elapsed_s=4526
2026-05-13 15:55:34 [INFO] iter=1400 val_acc=0.0377 best=0.0429 @ 200
2026-05-13 15:56:13 [INFO] iter=1425 loss=3.8055e+00 elapsed_s=4761
2026-05-13 15:56:52 [INFO] iter=1450 loss=3.7227e+00 elapsed_s=4800
2026-05-13 15:57:32 [INFO] iter=1475 loss=3.3838e+00 elapsed_s=4840
2026-05-13 15:58:11 [INFO] iter=1500 loss=3.7005e+00 elapsed_s=4879
2026-05-13 16:38:35 [INFO] iter=1500 val_acc=0.0405 best=0.0429 @ 200
2026-05-13 16:39:15 [INFO] iter=1525 loss=3.2409e+00 elapsed_s=7342
2026-05-13 16:39:54 [INFO] iter=1550 loss=3.3985e+00 elapsed_s=7382
2026-05-13 16:40:34 [INFO] iter=1575 loss=3.5714e+00 elapsed_s=7422
2026-05-13 16:41:13 [INFO] iter=1600 loss=3.5260e+00 elapsed_s=7461
2026-05-13 16:44:16 [INFO] iter=1600 val_acc=0.0405 best=0.0429 @ 200
2026-05-13 16:44:55 [INFO] iter=1625 loss=3.6518e+00 elapsed_s=7683
2026-05-13 16:45:35 [INFO] iter=1650 loss=3.5278e+00 elapsed_s=7723
2026-05-13 16:46:12 [INFO] iter=1675 loss=3.4389e+00 elapsed_s=7760
2026-05-13 16:46:51 [INFO] iter=1700 loss=3.6660e+00 elapsed_s=7799
2026-05-13 16:49:48 [INFO] iter=1700 val_acc=0.0405 best=0.0429 @ 200
2026-05-13 16:50:28 [INFO] iter=1725 loss=3.4282e+00 elapsed_s=8015
2026-05-13 16:51:06 [INFO] iter=1750 loss=3.7107e+00 elapsed_s=8054
2026-05-13 16:51:45 [INFO] iter=1775 loss=3.4735e+00 elapsed_s=8093
2026-05-13 16:52:23 [INFO] iter=1800 loss=3.7089e+00 elapsed_s=8131
2026-05-13 16:55:19 [INFO] iter=1800 val_acc=0.0405 best=0.0429 @ 200
2026-05-13 16:55:58 [INFO] iter=1825 loss=3.5783e+00 elapsed_s=8346
2026-05-13 16:56:37 [INFO] iter=1850 loss=3.6019e+00 elapsed_s=8385
2026-05-13 16:57:16 [INFO] iter=1875 loss=3.5512e+00 elapsed_s=8423
2026-05-13 16:57:55 [INFO] iter=1900 loss=3.4789e+00 elapsed_s=8463
2026-05-14 17:05:00 [INFO] Logging to /logs/training.log
2026-05-14 17:05:00 [INFO] WORKDIR=/workspace LOG_DIR=/logs OUT=/workspace/seminar_12_runs
2026-05-14 17:05:00 [INFO] torch 2.12.0+cpu cuda=False ME.cuda=False
2026-05-14 17:05:00 [INFO] Start training: max_steps=5000
2026-05-14 17:05:08 [INFO] iter=0 loss=3.8865e+00 elapsed_s=2
2026-05-14 17:05:46 [INFO] iter=25 loss=8.1931e+00 elapsed_s=41
2026-05-14 17:06:25 [INFO] iter=50 loss=4.0927e+00 elapsed_s=80
2026-05-14 17:07:05 [INFO] iter=75 loss=3.6264e+00 elapsed_s=119
2026-05-14 17:07:46 [INFO] iter=100 loss=3.4170e+00 elapsed_s=160
2026-05-14 17:10:48 [INFO] iter=100 val_acc=0.0373 best=0.0000 @ -1
2026-05-14 17:11:31 [INFO] iter=125 loss=3.2746e+00 elapsed_s=386
2026-05-14 17:12:13 [INFO] iter=150 loss=3.6347e+00 elapsed_s=428
2026-05-14 17:12:56 [INFO] iter=175 loss=3.6843e+00 elapsed_s=471
2026-05-14 17:13:38 [INFO] iter=200 loss=3.3845e+00 elapsed_s=512
2026-05-14 17:16:31 [INFO] iter=200 val_acc=0.0413 best=0.0373 @ 100
2026-05-14 17:17:11 [INFO] iter=225 loss=3.5494e+00 elapsed_s=725
2026-05-14 17:17:50 [INFO] iter=250 loss=3.6737e+00 elapsed_s=765
2026-05-14 17:18:29 [INFO] iter=275 loss=3.4182e+00 elapsed_s=803
2026-05-14 17:19:07 [INFO] iter=300 loss=3.4001e+00 elapsed_s=842
2026-05-14 17:21:58 [INFO] iter=300 val_acc=0.0393 best=0.0413 @ 200
2026-05-14 17:22:38 [INFO] iter=325 loss=3.6157e+00 elapsed_s=1052
2026-05-14 17:23:19 [INFO] iter=350 loss=3.6540e+00 elapsed_s=1094
2026-05-14 17:24:00 [INFO] iter=375 loss=3.4301e+00 elapsed_s=1134
2026-05-14 17:24:40 [INFO] iter=400 loss=3.4742e+00 elapsed_s=1175
2026-05-14 17:27:41 [INFO] iter=400 val_acc=0.0405 best=0.0413 @ 200
2026-05-14 17:28:24 [INFO] iter=425 loss=3.6277e+00 elapsed_s=1398
2026-05-14 17:29:06 [INFO] iter=450 loss=3.4758e+00 elapsed_s=1440
2026-05-14 17:29:47 [INFO] iter=475 loss=3.6387e+00 elapsed_s=1481
2026-05-14 17:30:26 [INFO] iter=500 loss=3.6603e+00 elapsed_s=1521
2026-05-14 17:33:22 [INFO] iter=500 val_acc=0.0405 best=0.0413 @ 200
2026-05-14 17:34:04 [INFO] iter=525 loss=3.6465e+00 elapsed_s=1739
2026-05-14 17:34:46 [INFO] iter=550 loss=3.7273e+00 elapsed_s=1780
2026-05-14 17:35:27 [INFO] iter=575 loss=3.5203e+00 elapsed_s=1822
2026-05-14 17:36:08 [INFO] iter=600 loss=3.5103e+00 elapsed_s=1863
2026-05-14 17:39:07 [INFO] iter=600 val_acc=0.0405 best=0.0413 @ 200
2026-05-14 17:39:48 [INFO] iter=625 loss=3.6223e+00 elapsed_s=2082
2026-05-14 17:40:30 [INFO] iter=650 loss=3.8184e+00 elapsed_s=2125
2026-05-14 17:41:11 [INFO] iter=675 loss=3.6826e+00 elapsed_s=2165
2026-05-14 17:41:52 [INFO] iter=700 loss=3.4881e+00 elapsed_s=2206
2026-05-14 17:44:54 [INFO] iter=700 val_acc=0.0405 best=0.0413 @ 200
2026-05-14 17:45:38 [INFO] iter=725 loss=3.7741e+00 elapsed_s=2432
2026-05-14 17:46:20 [INFO] iter=750 loss=3.6578e+00 elapsed_s=2474
2026-05-14 17:47:01 [INFO] iter=775 loss=3.5799e+00 elapsed_s=2515
2026-05-14 17:47:42 [INFO] iter=800 loss=3.6780e+00 elapsed_s=2556
2026-05-14 17:50:34 [INFO] iter=800 val_acc=0.0405 best=0.0413 @ 200
2026-05-14 17:51:13 [INFO] iter=825 loss=3.4450e+00 elapsed_s=2767
2026-05-14 17:51:52 [INFO] iter=850 loss=3.8453e+00 elapsed_s=2806
2026-05-14 17:52:31 [INFO] iter=875 loss=3.5829e+00 elapsed_s=2846
2026-05-14 17:53:10 [INFO] iter=900 loss=3.8018e+00 elapsed_s=2885
2026-05-14 17:56:04 [INFO] iter=900 val_acc=0.0442 best=0.0413 @ 200
2026-05-14 17:56:44 [INFO] iter=925 loss=3.3592e+00 elapsed_s=3098
2026-05-14 17:57:23 [INFO] iter=950 loss=3.5382e+00 elapsed_s=3138
2026-05-14 17:58:02 [INFO] iter=975 loss=3.5254e+00 elapsed_s=3177
2026-05-14 17:58:41 [INFO] iter=1000 loss=3.3534e+00 elapsed_s=3216
2026-05-14 18:01:35 [INFO] iter=1000 val_acc=0.0413 best=0.0442 @ 900
2026-05-14 18:02:15 [INFO] iter=1025 loss=3.8571e+00 elapsed_s=3430
2026-05-14 18:02:54 [INFO] iter=1050 loss=3.4339e+00 elapsed_s=3469
2026-05-14 18:03:34 [INFO] iter=1075 loss=3.4651e+00 elapsed_s=3508
2026-05-14 18:04:13 [INFO] iter=1100 loss=3.5085e+00 elapsed_s=3548
2026-05-14 18:07:07 [INFO] iter=1100 val_acc=0.0401 best=0.0442 @ 900
2026-05-14 18:07:45 [INFO] iter=1125 loss=3.7146e+00 elapsed_s=3760
2026-05-14 18:08:23 [INFO] iter=1150 loss=4.0333e+00 elapsed_s=3798
2026-05-14 18:09:01 [INFO] iter=1175 loss=3.6649e+00 elapsed_s=3836
2026-05-14 18:09:40 [INFO] iter=1200 loss=3.4316e+00 elapsed_s=3874
2026-05-14 18:12:31 [INFO] iter=1200 val_acc=0.0393 best=0.0442 @ 900
2026-05-14 18:13:10 [INFO] iter=1225 loss=3.4670e+00 elapsed_s=4084
2026-05-14 18:13:53 [INFO] iter=1250 loss=3.3460e+00 elapsed_s=4128
2026-05-14 18:14:31 [INFO] iter=1275 loss=3.7076e+00 elapsed_s=4166
2026-05-14 18:15:09 [INFO] iter=1300 loss=3.6004e+00 elapsed_s=4204
2026-05-14 18:18:02 [INFO] iter=1300 val_acc=0.0405 best=0.0442 @ 900
2026-05-14 18:18:43 [INFO] iter=1325 loss=3.7334e+00 elapsed_s=4417
2026-05-14 18:19:24 [INFO] iter=1350 loss=3.6362e+00 elapsed_s=4459
2026-05-14 18:20:05 [INFO] iter=1375 loss=3.1421e+00 elapsed_s=4500
2026-05-14 18:20:45 [INFO] iter=1400 loss=3.6804e+00 elapsed_s=4539
2026-05-14 18:23:38 [INFO] iter=1400 val_acc=0.0405 best=0.0442 @ 900
2026-05-14 18:24:19 [INFO] iter=1425 loss=3.7958e+00 elapsed_s=4754
2026-05-14 18:25:01 [INFO] iter=1450 loss=3.6945e+00 elapsed_s=4795
2026-05-14 18:25:42 [INFO] iter=1475 loss=3.3882e+00 elapsed_s=4837
2026-05-14 18:26:23 [INFO] iter=1500 loss=3.6937e+00 elapsed_s=4878
2026-05-14 18:29:17 [INFO] iter=1500 val_acc=0.0401 best=0.0442 @ 900
2026-05-14 18:29:56 [INFO] iter=1525 loss=3.2209e+00 elapsed_s=5090
2026-05-14 18:30:34 [INFO] iter=1550 loss=3.4048e+00 elapsed_s=5129
2026-05-14 18:31:12 [INFO] iter=1575 loss=3.5691e+00 elapsed_s=5167
2026-05-14 18:31:51 [INFO] iter=1600 loss=3.5308e+00 elapsed_s=5206
2026-05-14 18:34:43 [INFO] iter=1600 val_acc=0.0397 best=0.0442 @ 900
2026-05-14 18:35:23 [INFO] iter=1625 loss=3.6567e+00 elapsed_s=5417
2026-05-14 18:36:01 [INFO] iter=1650 loss=3.5310e+00 elapsed_s=5456
2026-05-14 18:36:38 [INFO] iter=1675 loss=3.4175e+00 elapsed_s=5493
2026-05-14 18:37:16 [INFO] iter=1700 loss=3.6595e+00 elapsed_s=5530
2026-05-14 18:40:04 [INFO] iter=1700 val_acc=0.0397 best=0.0442 @ 900
2026-05-14 18:40:43 [INFO] iter=1725 loss=3.4265e+00 elapsed_s=5737
2026-05-14 18:41:22 [INFO] iter=1750 loss=3.7169e+00 elapsed_s=5776
2026-05-14 18:42:00 [INFO] iter=1775 loss=3.4778e+00 elapsed_s=5815
2026-05-14 18:42:39 [INFO] iter=1800 loss=3.6970e+00 elapsed_s=5854
2026-05-14 18:45:31 [INFO] iter=1800 val_acc=0.0405 best=0.0442 @ 900
2026-05-14 18:46:09 [INFO] iter=1825 loss=3.5624e+00 elapsed_s=6063
2026-05-14 18:46:47 [INFO] iter=1850 loss=3.6585e+00 elapsed_s=6102
2026-05-14 18:47:26 [INFO] iter=1875 loss=3.5627e+00 elapsed_s=6140
2026-05-14 18:48:03 [INFO] iter=1900 loss=3.4963e+00 elapsed_s=6177
2026-05-14 18:50:53 [INFO] iter=1900 val_acc=0.0393 best=0.0442 @ 900
2026-05-14 18:51:31 [INFO] iter=1925 loss=3.7325e+00 elapsed_s=6385
2026-05-14 18:52:09 [INFO] iter=1950 loss=3.6378e+00 elapsed_s=6424
2026-05-14 18:52:48 [INFO] iter=1975 loss=3.7377e+00 elapsed_s=6463
2026-05-14 18:53:28 [INFO] iter=2000 loss=3.4862e+00 elapsed_s=6502
2026-05-14 18:56:22 [INFO] iter=2000 val_acc=0.0450 best=0.0442 @ 900
2026-05-14 18:57:05 [INFO] iter=2025 loss=3.6169e+00 elapsed_s=6719
2026-05-14 18:57:43 [INFO] iter=2050 loss=3.4770e+00 elapsed_s=6758
2026-05-14 18:58:23 [INFO] iter=2075 loss=3.4824e+00 elapsed_s=6797
2026-05-14 18:59:01 [INFO] iter=2100 loss=3.4480e+00 elapsed_s=6836
2026-05-14 19:01:59 [INFO] iter=2100 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 19:02:39 [INFO] iter=2125 loss=3.5786e+00 elapsed_s=7054
2026-05-14 19:03:17 [INFO] iter=2150 loss=3.6504e+00 elapsed_s=7092
2026-05-14 19:03:55 [INFO] iter=2175 loss=3.6683e+00 elapsed_s=7130
2026-05-14 19:04:34 [INFO] iter=2200 loss=3.2776e+00 elapsed_s=7169
2026-05-14 19:07:25 [INFO] iter=2200 val_acc=0.0425 best=0.0450 @ 2000
2026-05-14 19:08:04 [INFO] iter=2225 loss=3.5742e+00 elapsed_s=7378
2026-05-14 19:08:42 [INFO] iter=2250 loss=3.3910e+00 elapsed_s=7417
2026-05-14 19:09:21 [INFO] iter=2275 loss=3.5603e+00 elapsed_s=7455
2026-05-14 19:09:59 [INFO] iter=2300 loss=3.6400e+00 elapsed_s=7494
2026-05-14 19:12:51 [INFO] iter=2300 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 19:13:30 [INFO] iter=2325 loss=3.6971e+00 elapsed_s=7704
2026-05-14 19:14:08 [INFO] iter=2350 loss=3.5103e+00 elapsed_s=7743
2026-05-14 19:14:46 [INFO] iter=2375 loss=3.4967e+00 elapsed_s=7781
2026-05-14 19:15:25 [INFO] iter=2400 loss=3.4390e+00 elapsed_s=7819
2026-05-14 19:18:16 [INFO] iter=2400 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 19:18:54 [INFO] iter=2425 loss=3.6668e+00 elapsed_s=8029
2026-05-14 19:19:33 [INFO] iter=2450 loss=3.4662e+00 elapsed_s=8068
2026-05-14 19:20:19 [INFO] iter=2475 loss=3.2624e+00 elapsed_s=8113
2026-05-14 19:20:58 [INFO] iter=2500 loss=3.2862e+00 elapsed_s=8152
2026-05-14 19:23:50 [INFO] iter=2500 val_acc=0.0381 best=0.0450 @ 2000
2026-05-14 19:24:28 [INFO] iter=2525 loss=3.1604e+00 elapsed_s=8362
2026-05-14 19:25:04 [INFO] iter=2550 loss=3.5567e+00 elapsed_s=8399
2026-05-14 19:25:43 [INFO] iter=2575 loss=3.7038e+00 elapsed_s=8437
2026-05-14 19:26:21 [INFO] iter=2600 loss=3.4285e+00 elapsed_s=8475
2026-05-14 19:29:14 [INFO] iter=2600 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 19:29:53 [INFO] iter=2625 loss=3.9219e+00 elapsed_s=8688
2026-05-14 19:30:34 [INFO] iter=2650 loss=3.4265e+00 elapsed_s=8729
2026-05-14 19:31:20 [INFO] iter=2675 loss=3.5450e+00 elapsed_s=8775
2026-05-14 19:32:02 [INFO] iter=2700 loss=3.4907e+00 elapsed_s=8816
2026-05-14 19:35:04 [INFO] iter=2700 val_acc=0.0389 best=0.0450 @ 2000
2026-05-14 19:35:46 [INFO] iter=2725 loss=3.5598e+00 elapsed_s=9040
2026-05-14 19:36:26 [INFO] iter=2750 loss=3.5886e+00 elapsed_s=9081
2026-05-14 19:37:08 [INFO] iter=2775 loss=3.5848e+00 elapsed_s=9123
2026-05-14 19:37:50 [INFO] iter=2800 loss=3.4271e+00 elapsed_s=9164
2026-05-14 19:41:08 [INFO] iter=2800 val_acc=0.0397 best=0.0450 @ 2000
2026-05-14 19:41:47 [INFO] iter=2825 loss=3.3065e+00 elapsed_s=9401
2026-05-14 19:42:26 [INFO] iter=2850 loss=3.7069e+00 elapsed_s=9440
2026-05-14 19:43:04 [INFO] iter=2875 loss=3.5148e+00 elapsed_s=9479
2026-05-14 19:43:42 [INFO] iter=2900 loss=3.6513e+00 elapsed_s=9517
2026-05-14 19:46:37 [INFO] iter=2900 val_acc=0.0401 best=0.0450 @ 2000
2026-05-14 19:47:16 [INFO] iter=2925 loss=3.5468e+00 elapsed_s=9730
2026-05-14 19:47:54 [INFO] iter=2950 loss=3.4262e+00 elapsed_s=9769
2026-05-14 19:48:33 [INFO] iter=2975 loss=3.3586e+00 elapsed_s=9807
2026-05-14 19:49:13 [INFO] iter=3000 loss=3.7225e+00 elapsed_s=9847
2026-05-14 19:52:19 [INFO] iter=3000 val_acc=0.0397 best=0.0450 @ 2000
2026-05-14 19:53:00 [INFO] iter=3025 loss=3.3790e+00 elapsed_s=10075
2026-05-14 19:53:43 [INFO] iter=3050 loss=3.6563e+00 elapsed_s=10118
2026-05-14 19:54:26 [INFO] iter=3075 loss=3.6538e+00 elapsed_s=10161
2026-05-14 19:55:09 [INFO] iter=3100 loss=3.7655e+00 elapsed_s=10203
2026-05-14 19:58:23 [INFO] iter=3100 val_acc=0.0393 best=0.0450 @ 2000
2026-05-14 19:59:03 [INFO] iter=3125 loss=3.4894e+00 elapsed_s=10437
2026-05-14 19:59:41 [INFO] iter=3150 loss=3.7161e+00 elapsed_s=10476
2026-05-14 20:00:19 [INFO] iter=3175 loss=3.4674e+00 elapsed_s=10514
2026-05-14 20:00:58 [INFO] iter=3200 loss=3.5139e+00 elapsed_s=10553
2026-05-14 20:03:55 [INFO] iter=3200 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 20:04:34 [INFO] iter=3225 loss=3.3411e+00 elapsed_s=10768
2026-05-14 20:05:12 [INFO] iter=3250 loss=3.7022e+00 elapsed_s=10806
2026-05-14 20:05:49 [INFO] iter=3275 loss=3.4735e+00 elapsed_s=10844
2026-05-14 20:06:28 [INFO] iter=3300 loss=3.8888e+00 elapsed_s=10882
2026-05-14 20:09:25 [INFO] iter=3300 val_acc=0.0401 best=0.0450 @ 2000
2026-05-14 20:10:05 [INFO] iter=3325 loss=3.6019e+00 elapsed_s=11099
2026-05-14 20:10:45 [INFO] iter=3350 loss=3.7498e+00 elapsed_s=11140
2026-05-14 20:11:24 [INFO] iter=3375 loss=3.6170e+00 elapsed_s=11179
2026-05-14 20:12:02 [INFO] iter=3400 loss=3.7202e+00 elapsed_s=11216
2026-05-14 20:14:58 [INFO] iter=3400 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 20:15:37 [INFO] iter=3425 loss=3.4727e+00 elapsed_s=11431
2026-05-14 20:16:14 [INFO] iter=3450 loss=3.4221e+00 elapsed_s=11469
2026-05-14 20:16:51 [INFO] iter=3475 loss=3.3365e+00 elapsed_s=11505
2026-05-14 20:17:29 [INFO] iter=3500 loss=3.4306e+00 elapsed_s=11543
2026-05-14 20:20:28 [INFO] iter=3500 val_acc=0.0401 best=0.0450 @ 2000
2026-05-14 20:21:06 [INFO] iter=3525 loss=3.6489e+00 elapsed_s=11761
2026-05-14 20:21:44 [INFO] iter=3550 loss=3.6864e+00 elapsed_s=11799
2026-05-14 20:22:23 [INFO] iter=3575 loss=3.4686e+00 elapsed_s=11837
2026-05-14 20:23:04 [INFO] iter=3600 loss=3.6329e+00 elapsed_s=11878
2026-05-14 20:26:06 [INFO] iter=3600 val_acc=0.0397 best=0.0450 @ 2000
2026-05-14 20:26:44 [INFO] iter=3625 loss=3.3956e+00 elapsed_s=12099
2026-05-14 20:27:22 [INFO] iter=3650 loss=3.6247e+00 elapsed_s=12136
2026-05-14 20:27:59 [INFO] iter=3675 loss=3.5345e+00 elapsed_s=12173
2026-05-14 20:28:48 [INFO] iter=3700 loss=3.4231e+00 elapsed_s=12223
2026-05-14 20:31:44 [INFO] iter=3700 val_acc=0.0393 best=0.0450 @ 2000
2026-05-14 20:32:23 [INFO] iter=3725 loss=3.5982e+00 elapsed_s=12437
2026-05-14 20:33:01 [INFO] iter=3750 loss=3.4956e+00 elapsed_s=12476
2026-05-14 20:33:39 [INFO] iter=3775 loss=3.3634e+00 elapsed_s=12513
2026-05-14 20:34:17 [INFO] iter=3800 loss=3.4262e+00 elapsed_s=12552
2026-05-14 20:37:14 [INFO] iter=3800 val_acc=0.0401 best=0.0450 @ 2000
2026-05-14 20:37:53 [INFO] iter=3825 loss=3.4865e+00 elapsed_s=12767
2026-05-14 20:38:31 [INFO] iter=3850 loss=3.6753e+00 elapsed_s=12805
2026-05-14 20:39:12 [INFO] iter=3875 loss=3.5888e+00 elapsed_s=12847
2026-05-14 20:39:55 [INFO] iter=3900 loss=3.5144e+00 elapsed_s=12889
2026-05-14 20:43:09 [INFO] iter=3900 val_acc=0.0393 best=0.0450 @ 2000
2026-05-14 20:43:51 [INFO] iter=3925 loss=3.7290e+00 elapsed_s=13125
2026-05-14 20:44:33 [INFO] iter=3950 loss=3.6218e+00 elapsed_s=13167
2026-05-14 20:45:17 [INFO] iter=3975 loss=3.6801e+00 elapsed_s=13212
2026-05-14 20:46:02 [INFO] iter=4000 loss=3.6402e+00 elapsed_s=13256
2026-05-14 20:49:15 [INFO] iter=4000 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 20:49:56 [INFO] iter=4025 loss=3.6956e+00 elapsed_s=13490
2026-05-14 20:50:37 [INFO] iter=4050 loss=3.6373e+00 elapsed_s=13531
2026-05-14 20:51:16 [INFO] iter=4075 loss=3.5887e+00 elapsed_s=13570
2026-05-14 20:51:55 [INFO] iter=4100 loss=3.9668e+00 elapsed_s=13609
2026-05-14 20:54:59 [INFO] iter=4100 val_acc=0.0397 best=0.0450 @ 2000
2026-05-14 20:55:45 [INFO] iter=4125 loss=3.6806e+00 elapsed_s=13839
2026-05-14 20:56:28 [INFO] iter=4150 loss=3.3045e+00 elapsed_s=13883
2026-05-14 20:57:08 [INFO] iter=4175 loss=3.7358e+00 elapsed_s=13922
2026-05-14 20:57:48 [INFO] iter=4200 loss=3.4768e+00 elapsed_s=13962
2026-05-14 21:00:46 [INFO] iter=4200 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 21:01:26 [INFO] iter=4225 loss=3.2953e+00 elapsed_s=14180
2026-05-14 21:02:06 [INFO] iter=4250 loss=3.4431e+00 elapsed_s=14220
2026-05-14 21:02:51 [INFO] iter=4275 loss=3.5375e+00 elapsed_s=14266
2026-05-14 21:03:34 [INFO] iter=4300 loss=3.5732e+00 elapsed_s=14308
2026-05-14 21:07:06 [INFO] iter=4300 val_acc=0.0397 best=0.0450 @ 2000
2026-05-14 21:07:53 [INFO] iter=4325 loss=3.4352e+00 elapsed_s=14567
2026-05-14 21:08:39 [INFO] iter=4350 loss=3.5890e+00 elapsed_s=14613
2026-05-14 21:09:25 [INFO] iter=4375 loss=3.5790e+00 elapsed_s=14659
2026-05-14 21:10:10 [INFO] iter=4400 loss=3.5804e+00 elapsed_s=14704
2026-05-14 21:13:42 [INFO] iter=4400 val_acc=0.0405 best=0.0450 @ 2000
2026-05-14 21:14:29 [INFO] iter=4425 loss=3.5402e+00 elapsed_s=14963
2026-05-14 21:15:14 [INFO] iter=4450 loss=3.4093e+00 elapsed_s=15009
2026-05-14 21:16:00 [INFO] iter=4475 loss=3.5299e+00 elapsed_s=15055
2026-05-14 21:16:46 [INFO] iter=4500 loss=3.6095e+00 elapsed_s=15101
2026-05-14 21:20:24 [INFO] iter=4500 val_acc=0.0401 best=0.0450 @ 2000
2026-05-14 21:21:09 [INFO] iter=4525 loss=3.6632e+00 elapsed_s=15364
2026-05-14 21:21:55 [INFO] iter=4550 loss=3.5558e+00 elapsed_s=15409
2026-05-14 21:22:40 [INFO] iter=4575 loss=3.2250e+00 elapsed_s=15454
2026-05-14 21:23:19 [INFO] iter=4600 loss=3.5909e+00 elapsed_s=15494
2026-05-14 21:26:20 [INFO] iter=4600 val_acc=0.0397 best=0.0450 @ 2000
2026-05-14 21:26:58 [INFO] iter=4625 loss=3.4307e+00 elapsed_s=15712
2026-05-14 21:27:36 [INFO] iter=4650 loss=3.4443e+00 elapsed_s=15751
2026-05-14 21:28:14 [INFO] iter=4675 loss=3.5435e+00 elapsed_s=15789
2026-05-14 21:28:52 [INFO] iter=4700 loss=3.5206e+00 elapsed_s=15827
2026-05-14 21:31:49 [INFO] iter=4700 val_acc=0.0393 best=0.0450 @ 2000
2026-05-14 21:32:27 [INFO] iter=4725 loss=3.5418e+00 elapsed_s=16041
2026-05-14 21:33:04 [INFO] iter=4750 loss=3.3549e+00 elapsed_s=16079
2026-05-14 21:33:43 [INFO] iter=4775 loss=3.4396e+00 elapsed_s=16117
2026-05-14 21:34:22 [INFO] iter=4800 loss=3.6116e+00 elapsed_s=16156
2026-05-14 21:37:20 [INFO] iter=4800 val_acc=0.0401 best=0.0450 @ 2000
2026-05-14 21:37:58 [INFO] iter=4825 loss=3.6384e+00 elapsed_s=16373
2026-05-14 21:38:36 [INFO] iter=4850 loss=3.5439e+00 elapsed_s=16411
2026-05-14 21:39:14 [INFO] iter=4875 loss=3.6435e+00 elapsed_s=16449
2026-05-14 21:39:52 [INFO] iter=4900 loss=3.7249e+00 elapsed_s=16487
2026-05-14 21:42:53 [INFO] iter=4900 val_acc=0.0397 best=0.0450 @ 2000
2026-05-14 21:48:57 [INFO] iter=4925 loss=3.8172e+00 elapsed_s=17032
2026-05-14 21:49:37 [INFO] iter=4950 loss=3.5808e+00 elapsed_s=17071
2026-05-14 21:50:14 [INFO] iter=4975 loss=3.6745e+00 elapsed_s=17109
2026-05-14 21:50:54 [INFO] Saved history.json, /workspace/seminar_12_runs/metrics.png, best_model.pth in /workspace/seminar_12_runs
2026-05-14 21:53:56 [INFO] Final test_acc=0.0393 best_val=0.0450 @ step 2000


Получилось accuracy в 3.9%, что ужасно, однако лучше чем случайный выбор, который даст только 2.5%. С учетом того, что пришлось учить на CPU, 5000 шагов достаточно для отрыва от качества случайного выбора.

Стоит заметить, что запускался код из `seminars/seminar_12/minkowski_training_docker/Dockerfile.cpu`